## Prediction of Customer Churn Pbobability

**Version 1.0**

The notebook in fourth entry to competition I started with Basics to start with using Random Fore7r, before I test pipelione and ensemble

**Version 2.0**

Random Forests tend to struggle with this because they average predictions from many trees, which often pushes probabilities away from 0 and 1 toward the center.
This version introduces Random Forest Calibration By applying Platt Scaling (Sigmoid calibration), turns a mathematical output into a reliable financial metric for churn prevention strategies.

**Version 3.0**

Introduced GridSearchCV: It no longer guesses the number of trees; it tests 50, 100, 150, 200, 250 and 300 to find the most accurate estimator for specific data.

**Version 4.0**

Introduced notebook pipeline using Balanced Random Forest (from imblearn) designed to:
- Handle churn imbalance properly
- Increase churn probability sensitivity
- Optimize for ROC-AUC
- Improve recall for churners
- Provide calibrated probabilities
- Use stratified CV
- Automatically tune threshold

**Version 5.0***

Added multi-seed ensemble combining XGBoost + Random Forest, optimized for churn imbalance and probability separation.
- XGBoost with scale_pos_weight
- RandomForest with class_weight='balanced'
- 3 seeds
- 5-fold stratified CV
- Rank normalization blending
- Weight optimization
- Threshold tuning for recall

**Version 6.0**

Added CatBoost to XGBoost and RF
  3-model multi-seed ensemble:
- XGBoost
- Random Forest
- CatBoost
- 3 seeds
- 5-fold stratified CV
- Rank blending
- Weight optimization
- Threshold tuning

**Version 7.0**


Added LightGBM to the previous XGBoost + Random Forest + CatBoost multi-seed ensemble to evaluate 4 model ensemble, these four specific algorithms leverage different mathematical "perspectives" on your data:
- XGBoost & LightGBM: These are "Gradient Boosted Decision Trees" (GBDT). They build trees sequentially, where each tree tries to correct the errors of the previous one. They are highly aggressive and excellent at finding complex patterns.
- CatBoost: It uses a specialized technique called "Ordered Boosting" which is specifically designed to reduce bias when dealing with categorical variables. It often finds insights that XGB/LGB miss.
- Random Forest: This is a "Bagging" model. Unlike the others, it builds many independent trees in parallel and averages them. It is much harder to overfit, making it a perfect "anchor" for your ensemble to prevent the boosting models from becoming too volatile.

**Version 8.0**

Added Feature Engineering along with automatic ensemble weight calculation, the model now has
- Feature Engineering (tenure ratios, charge ratios, service counts)
- Automatic Ensemble Weight Search
- Multi-seed ensemble (XGB + LGBM + CatBoost + RF)
- Rank blending
- Grid search for optimal ensemble weights

**Version 9.0**

Improved pipeline by adding
- Automatic feature generation
- Multi-seed base models (XGB + LGBM + CatBoost + RF)
- Out-of-fold predictions
- Stacking meta-model (Logistic Regression)
- Rank normalization

The pipeline can be depicted:

- Raw Data
- Feature Engineering
- Base Models (Level-1)

--   ├── XGBoost  
--   ├── LightGBM  
--   ├── CatBoost
--   └── RandomForest

- OOF Predictions
- Stacking Dataset
- Meta Model (Level-2)
- Logistic Regression
- Final Predictions

**Version 10.0**
 - Raw Data
 - Feature Engineering
 - Encoding + Scaling
 - Level-1 Models
   -- ── XGBoost
   -- ── LightGBM
   -- ── CatBoost
 - OOF Predictions
 - Pseudo-Labeling
 - Level-2 Stacking
   -- ── Ridge
   -- ── Logistic
   -- ── Neural Network
 - Final Ensemble

**Version 11.0**
 - Raw Data
 - Feature Engineering
 - Encoding + Scaling
 - Level-1 Multi Seed Models 
   -- ── XGBoost
   -- ── LightGBM
   -- ── CatBoost
 - OOF Predictions
 - Pseudo-Labeling
 - Level-2 Stacking
   -- ── Ridge
   -- ── Logistic
   -- ── Neural Network
 - Final Ensemble

**Version 12.0**

Added auto ensemble for level 2 stacking

In [1]:
# ============================================================
# 1. Imports
# ============================================================
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
import warnings

# Suppress the specific Pandas FutureWarning
warnings.filterwarnings("ignore", category=FutureWarning, message=".*Setting an item of incompatible dtype.*")

# Suppress the Scikit-learn UserWarning about feature names
warnings.filterwarnings("ignore", category=UserWarning, message=".*X does not have valid feature names.*")

In [2]:
# ============================================================
# 2. Configuration & Data Loading
# ============================================================
SEEDS = [42, 2023, 888] # Changed one seed for diversity
N_FOLDS = 5
TARGET = "Churn"
ID = "id"

train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")
test_ids = test[ID]

# Map target early
train[TARGET] = train[TARGET].map({"No": 0, "Yes": 1})

In [3]:
# ============================================================
# 3. Enhanced Feature Engineering
# ============================================================

def enhanced_feature_engineering(df):
    # Base ratios
    df["avg_monthly_charge"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["charge_ratio"] = df["MonthlyCharges"] / (df["TotalCharges"] + 1)
    
    # Service interaction features
    service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 
                    'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
    
    # Count total active services
    df['total_services'] = df[service_cols].apply(lambda x: x.str.contains('Yes|Fiber|DSL').sum(), axis=1)
    
    # Financial stress indicators
    df['high_cost_short_tenure'] = (df['MonthlyCharges'] > df['MonthlyCharges'].median()) & (df['tenure'] < df['tenure'].median())
    df['high_cost_short_tenure'] = df['high_cost_short_tenure'].astype(int)

    return df

train = enhanced_feature_engineering(train)
test = enhanced_feature_engineering(test)

# Categorical Encoding
combined = pd.concat([train, test])
cat_features = combined.select_dtypes(include=['object']).columns

for col in cat_features:
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col].astype(str))

train = combined.iloc[:len(train)]
test = combined.iloc[len(train):]

X = train.drop([TARGET, ID], axis=1)
y = train[TARGET]
X_test = test.drop([TARGET, ID], axis=1, errors="ignore")

In [4]:
# ============================================================
# 5. Initialize OOF container + Level 1 Training
# ============================================================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

model_names = ['XGB', 'LGB', 'CAT', 'HIST']
oof_preds = {name: np.zeros(len(X)) for name in model_names}
test_preds = {name: np.zeros(len(X_test)) for name in model_names}

# Dictionary to store fold results for summary
fold_results = {name: [] for name in model_names}

for seed in SEEDS:
    print(f"\n{'='*20} STARTING SEED {seed} {'='*20}")
    
    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        # Initialize Models
        m_xgb = xgb.XGBClassifier(n_estimators=5000, learning_rate=0.0125, max_depth=8, random_state=seed, tree_method="hist")
        m_lgb = lgb.LGBMClassifier(n_estimators=5000, learning_rate=0.0125, random_state=seed, verbose=-1)
        m_cat = CatBoostClassifier(iterations=5000, learning_rate=0.0125, depth=8, verbose=False, random_seed=seed)
        m_hist = HistGradientBoostingClassifier(max_iter=2500, learning_rate=0.02, random_state=seed)

        models = [m_xgb, m_lgb, m_cat, m_hist]
        
        # Print header for the fold
        print(f"Fold {fold}: ", end="")
        
        fold_scores = []
        for name, model in zip(model_names, models):
            model.fit(X_train, y_train)
            
            # Validation Predictions
            val_p = model.predict_proba(X_valid)[:, 1]
            test_p = model.predict_proba(X_test)[:, 1]
            
            # Calculate Fold Metric
            auc_score = roc_auc_score(y_valid, val_p)
            fold_results[name].append(auc_score)
            fold_scores.append(f"{name}: {auc_score:.4f}")
            
            # Update OOF and Test
            oof_preds[name][valid_idx] += val_p / len(SEEDS)
            test_preds[name] += test_p / (N_FOLDS * len(SEEDS))
        
        # Output results for this specific fold
        print(" | ".join(fold_scores))

# Summary of Level 1 Performance
print(f"\n{'='*20} LEVEL 1 SUMMARY {'='*20}")
for name in model_names:
    avg_auc = np.mean(fold_results[name])
    std_auc = np.std(fold_results[name])
    print(f"{name} Overall OOF AUC: {avg_auc:.5f} +/- {std_auc:.5f}")




==================== STARTING SEED 42 ====================
Fold 0: XGB: 0.9142 | LGB: 0.9157 | CAT: 0.9160 | HIST: 0.9154
Fold 1: XGB: 0.9153 | LGB: 0.9169 | CAT: 0.9172 | HIST: 0.9163
Fold 2: XGB: 0.9149 | LGB: 0.9162 | CAT: 0.9166 | HIST: 0.9156
Fold 3: XGB: 0.9158 | LGB: 0.9173 | CAT: 0.9175 | HIST: 0.9167
Fold 4: XGB: 0.9129 | LGB: 0.9145 | CAT: 0.9147 | HIST: 0.9141

==================== STARTING SEED 2023 ====================
Fold 0: XGB: 0.9142 | LGB: 0.9158 | CAT: 0.9160 | HIST: 0.9153
Fold 1: XGB: 0.9153 | LGB: 0.9169 | CAT: 0.9172 | HIST: 0.9165
Fold 2: XGB: 0.9149 | LGB: 0.9162 | CAT: 0.9166 | HIST: 0.9157
Fold 3: XGB: 0.9158 | LGB: 0.9171 | CAT: 0.9176 | HIST: 0.9169
Fold 4: XGB: 0.9129 | LGB: 0.9145 | CAT: 0.9148 | HIST: 0.9139

==================== STARTING SEED 888 ====================
Fold 0: XGB: 0.9142 | LGB: 0.9158 | CAT: 0.9161 | HIST: 0.9153
Fold 1: XGB: 0.9153 | LGB: 0.9170 | CAT: 0.9172 | HIST: 0.9164
Fold 2: XGB: 0.9149 | LGB: 0.9162 | CAT: 0.9167 | HIST: 0.915

In [5]:
# ============================================================
# 6. Rank Normalization
# ============================================================
def rank_norm(x): return rankdata(x) / len(x)

X_stack = np.column_stack([rank_norm(oof_preds[n]) for n in model_names])
X_test_stack = np.column_stack([rank_norm(test_preds[n]) for n in model_names])

In [6]:
# ============================================================
# 7. Level 2: Weighted Meta-Model with Performance Output
# ============================================================
from sklearn.model_selection import cross_val_predict

print(f"\n{'='*20} LEVEL 2 META-MODEL TRAINING {'='*20}")

meta_ridge = Ridge(alpha=0.5)
meta_log = LogisticRegression(C=1.0)

oof_meta_ridge = cross_val_predict(meta_ridge, X_stack, y, cv=5)
oof_meta_log = cross_val_predict(meta_log, X_stack, y, cv=5, method='predict_proba')[:, 1]

meta_ridge.fit(X_stack, y)
meta_log.fit(X_stack, y)

auc_ridge = roc_auc_score(y, oof_meta_ridge)
auc_log = roc_auc_score(y, oof_meta_log)

final_oof_blend = (rank_norm(oof_meta_ridge) * 0.4) + (rank_norm(oof_meta_log) * 0.6)
auc_final = roc_auc_score(y, final_oof_blend)

print(f"Meta-Ridge OOF AUC: {auc_ridge:.5f}")
print(f"Meta-Logistic OOF AUC: {auc_log:.5f}")
print(f"Final Stacking Blend OOF AUC: {auc_final:.5f}")

ridge_test_preds = meta_ridge.predict(X_test_stack)
log_test_preds = meta_log.predict_proba(X_test_stack)[:, 1]

final_test_probs = (rank_norm(ridge_test_preds) * 0.4) + (rank_norm(log_test_preds) * 0.6)

print(f"\n{'='*20} PROCESSING COMPLETE {'='*20}")
print(f"Final Prediction Mean: {final_test_probs.mean():.4f}")



==================== LEVEL 2 META-MODEL TRAINING ====================
Meta-Ridge OOF AUC: 0.91658
Meta-Logistic OOF AUC: 0.91655
Final Stacking Blend OOF AUC: 0.91658

==================== PROCESSING COMPLETE ====================
Final Prediction Mean: 0.5000


In [7]:
# ============================================================
# 8. Submission
# ============================================================
submission = pd.DataFrame({ID: test_ids, TARGET: final_test_probs})
submission.to_csv("submission.csv", index=False)

print("Stacking complete. Meta-model blending finished.")
submission.head(20)




Stacking complete. Meta-model blending finished.


,id,Churn
0,594194,0.523668
1,594195,0.024744
2,594196,0.553453
3,594197,0.163797
4,594198,0.802729
5,594199,0.629207
6,594200,0.985079
7,594201,0.157151
8,594202,0.418870
9,594203,0.712727
